# Ollama Basic Session Example (native API)

Minimal example against Ollama's **native** API, not the OpenAI-compat
layer. Two disjoint route trees exist on Ollama's own server
(`server/routes.go` in `ollama/ollama`) and this notebook only talks
to the native one:

- Native (this notebook): `/api/tags`, `/api/chat` -- no `/v1` anywhere
- OpenAI-compat (not used here): `/v1/chat/completions` -- this is
  what `opencode`'s `local/ollama` provider uses instead
  (`OPENCODE_OLLAMA_BASE_URL`, kept separate in `terraform/variables.tf`
  as `var.opencode_ollama_base_url`)

`OLLAMA_BASE_URL` here maps to `var.ollama_native_base_url` in
Terraform / the `jupyter` container's env -- same variable
`run_eval_client.py`'s warm-up/unload logic already uses.


In [1]:
import json
import os
import urllib.error
import urllib.request

OLLAMA_BASE_URL = os.environ.get("OLLAMA_BASE_URL", "http://host.docker.internal:11434")
OLLAMA_MODEL = os.environ.get("OLLAMA_MODEL", "qwen2.5-coder:7b")


def _request(method, path, body=None, timeout=60):
    trimmed = OLLAMA_BASE_URL.rstrip("/")
    url = f"{trimmed}{path}"
    data = json.dumps(body).encode("utf-8") if body is not None else None
    req = urllib.request.Request(url, data=data, method=method)
    try:
        with urllib.request.urlopen(req, timeout=timeout) as resp:
            raw = resp.read()
    except urllib.error.HTTPError as exc:
        detail = exc.read().decode("utf-8", "replace")
        raise RuntimeError(f"{method} {path} -> HTTP {exc.code}: {detail}") from exc
    return json.loads(raw) if raw else {}


def list_models():
    """GET /api/tags -- native endpoint, confirmed at
    ollama/ollama:server/routes.go (r.GET("/api/tags", s.ListHandler)).
    """
    return [m["name"] for m in _request("GET", "/api/tags").get("models", [])]


def chat(model, prompt, timeout=300):
    """POST /api/chat, non-streaming. stream: false avoids having to
    parse newline-delimited JSON chunks for this minimal example.
    """
    body = {
        "model": model,
        "messages": [{"role": "user", "content": prompt}],
        "stream": False,
    }
    response = _request("POST", "/api/chat", body, timeout=timeout)
    return response.get("message", {}).get("content", "")


In [2]:
from IPython.display import Markdown, display

available = list_models()
listing = "\n".join(f"- `{m}`" for m in available)
display(Markdown(f"### Models available on this Ollama instance\n{listing}"))

if OLLAMA_MODEL not in available:
    raise ValueError(f"{OLLAMA_MODEL!r} not found. Available: {available}")

reply = chat(OLLAMA_MODEL, "Hello")
display(Markdown("## User\n\nHello"))
display(Markdown(f"## Assistant\n\n{reply}"))


### Models available on this Ollama instance
- `qwen2.5-coder:7b`

## User

Hello

## Assistant

Hello from mock Ollama.